# Pipeline — Detecção de Grape Black Rot

**Processamento Digital de Imagens · Capstone**
Centro Universitário Dom Helder · Prof. Dr. Fischer Stefan

## Etapas do pipeline

| Etapa | Técnica | Por quê |
|---|---|---|
| 1 | **Gaussian Blur (5×5)** | Reduz ruído antes de toda operação |
| 2 | **Canny canal S + morfologia** | Isola a folha do fundo sem incluir sombra ou pecíolo |
| 3 | **HSV Marrom** | Detecta as manchas Black Rot dentro da folha |
| 4 | **Canny por ROI** | Refina a borda de cada lesão localizada |

> Execute **Run All**. Resultados salvos em `results/07_pipeline_grape/`.


---
## 0 · Setup

In [ ]:
%matplotlib inline
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

def _achar_root():
    root = Path.cwd()
    while root != root.parent:
        if (root / '.git').exists() or (root / 'requirements.txt').exists():
            break
        root = root.parent
    return root

ROOT = _achar_root()
sys.path.insert(0, str(ROOT / 'src'))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from preprocessing import (
    aplicar_blur_gaussiano, converter_grayscale,
    aplicar_blur_mediano, aplicar_blur_bilateral,
)
from detection import (
    detectar_bordas_canny, detectar_bordas_sobel,
    detectar_cantos_harris, detectar_cantos_shi_tomasi,
    detectar_contornos_externos,
    segmentar_folha_verde, detectar_lesoes_hsv,
    destacar_regiao_doenca, compor_diagnostico_visual,
)
from yolo_detection import casar_caixas
from dataset import (
    listar_classes, nome_legivel, ler_labels_yolo,
    carregar_imagem_rgb, amostrar_imagens, desenhar_boxes,
)
from utils import grade, sep

plt.rcParams.update({'figure.dpi': 100, 'figure.facecolor': 'white'})

TAMANHO  = (416, 416)
DATA_DIR = str(ROOT / 'data')
RD       = str(ROOT / 'results' / '07_pipeline_grape')
os.makedirs(RD, exist_ok=True)

CLASSES = listar_classes(DATA_DIR)
print('Setup OK')
print(f'ROOT     : {ROOT}')
print(f'DATA_DIR : {DATA_DIR}')
print(f'Saida    : {RD}')


---
## 1 · Amostra A — Grape Black Rot

In [ ]:
# Mesma seleção do notebook principal (semente=7) — garante reprodutibilidade
sel    = amostrar_imagens(DATA_DIR, n=12, split='train', com_label=True, semente=7)
PATH_A, PATH_B = sel[0], sel[1]
IMG_A  = cv2.resize(carregar_imagem_rgb(PATH_A), TAMANHO)
IMG_B  = cv2.resize(carregar_imagem_rgb(PATH_B), TAMANHO)
GT_A   = ler_labels_yolo(PATH_A, IMG_A.shape)
GT_B   = ler_labels_yolo(PATH_B, IMG_B.shape)

if len(GT_B) > len(GT_A):
    IMG_A, IMG_B = IMG_B, IMG_A
    GT_A,  GT_B  = GT_B,  GT_A
    PATH_A, PATH_B = PATH_B, PATH_A

NOME_A = ', '.join(sorted({nome_legivel(CLASSES[c[0]]) for c in GT_A})) or 'sem rotulo'
print(f'Arquivo  : {os.path.basename(PATH_A)}')
print(f'Classe   : {NOME_A}')
print(f'GT boxes : {len(GT_A)}')

grade(
    [desenhar_boxes(IMG_A, GT_A, CLASSES)],
    [f'A — {NOME_A} ({len(GT_A)} caixas GT)'],
    cols=1, figsize=(5, 5),
    nome='00_amostra_A.png', results_dir=RD,
)


---
## 2 · Etapa 1 — Gaussian Blur (5×5)

O blur é aplicado **antes de todas as etapas** do pipeline.
A comparação abaixo justifica a escolha do Gaussian 5×5:
- Reduz speckle sem destruir bordas das lesões
- Melhora a qualidade da máscara HSV
- Thresholds do Canny ficam mais estáveis


In [ ]:
# Comparação de configurações de blur na Amostra A
configs = [
    ('Original',      IMG_A),
    ('Gaussian 3×3',  aplicar_blur_gaussiano(IMG_A, kernel=(3, 3))),
    ('Gaussian 5×5',  aplicar_blur_gaussiano(IMG_A, kernel=(5, 5))),
    ('Gaussian 7×7',  aplicar_blur_gaussiano(IMG_A, kernel=(7, 7))),
    ('Median k=3',    aplicar_blur_mediano(IMG_A,   ksize=3)),
    ('Bilateral',     aplicar_blur_bilateral(IMG_A)),
]
grade(
    [img for _, img in configs],
    [tit for tit, _ in configs],
    cols=3, figsize=(15, 10),
    nome='01_blur_comparacao.png', results_dir=RD,
)

sep('Análise: redução de ruído por blur')
gray_orig = converter_grayscale(IMG_A)
for tit, img in configs[1:]:
    g   = converter_grayscale(img)
    red = (1 - g.std() / gray_orig.std()) * 100
    print(f'  {tit:<16}: std={g.std():.2f}  reducao={red:.1f}%')
print()
print('Escolha: Gaussian 5×5 — bom balanço entre redução de ruído e preservação de bordas.')

# Versão pré-processada usada em todo o resto
IMG_A_B = aplicar_blur_gaussiano(IMG_A, kernel=(5, 5))
print('\nIMG_A_B criado (Gaussian 5×5) — usado em todas as etapas seguintes.')


---
## 3 · Etapa 2 — Edge & Corner Detection (entrada: `IMG_A_B`)

Todas as técnicas recebem a imagem já blurrada.
**Conclusão antecipada:** Canny é o melhor para Black Rot —
detecta a borda nítida entre a lesão escura e o tecido saudável.


In [ ]:
canny_a      = detectar_bordas_canny(IMG_A_B)
mag, sx, sy  = detectar_bordas_sobel(IMG_A_B)
img_h, n_h   = detectar_cantos_harris(IMG_A_B)
img_s, n_s   = detectar_cantos_shi_tomasi(IMG_A_B, max_cantos=120)
cont_a, n_ca = detectar_contornos_externos(IMG_A_B)

grade(
    [IMG_A_B,  canny_a,  mag,     img_h,           img_s,              cont_a     ],
    ['Blur',  'Canny',  'Sobel', f'Harris ({n_h})', f'Shi-T. ({n_s})', f'Cont. ({n_ca})'],
    cols=6, figsize=(22, 4),
    nome='02_detectores.png', results_dir=RD,
)

sep('Análise: detectores de borda/canto')
n_bordas = int((canny_a > 0).sum())
total_px = TAMANHO[0] * TAMANHO[1]
med      = int(np.median(converter_grayscale(IMG_A_B)))
print(f'  Pixels Canny      : {n_bordas:,}  ({n_bordas/total_px*100:.1f}%)')
print(f'  Magnitude Sobel   : {mag.mean():.2f} (média)')
print(f'  Cantos Harris     : {n_h:,}')
print(f'  Cantos Shi-Tomasi : {n_s}  (limite=120)')
print(f'  Contornos externos: {n_ca}')
print()
print('Canny será reutilizado no Passo 4, mas SÓ dentro das ROIs do HSV.')
print('Isso elimina o ruído das nervuras e mantém apenas as bordas das lesões.')


---
## 4 · Etapa 3a — Máscara da Folha (Canny canal S)

Entrada: `IMG_A_B` (já blurrada).

Pipeline completo para isolar a folha sem incluir sombra ou pecíolo:

1. **Canny no canal S (saturação):** folha colorida = S alto; fundo cinza e sombra = S ≈ 0.
   A borda folha/fundo é muito mais nítida no canal S do que no grayscale.
   Thresholds fixos: low=20, high=60.
2. **Dilata (5×5, 3 it.) → `fillPoly` do maior contorno** → limite externo rígido da folha.
3. **Remove pixels acrômáticos** (S<30, V>80) que ficam nas reentrâncias da borda recortada.
4. **Erosão grande (25×25) para desconectar o pecíolo** (~20-30 px de largura);
   mantém apenas o maior componente (corpo da folha); dilata de volta dentro do limite Canny.
5. **CLOSE (7×7, 2 it.)** fecha os buracos escuros das lesões no interior da folha.


In [ ]:
# MÁSCARA FOLHA — Canny canal S + morfologia
folha_seg, mascara_folha = segmentar_folha_verde(IMG_A_B)

grade(
    [IMG_A,      IMG_A_B,          mascara_folha,           folha_seg          ],
    ['Original', 'Pré-Blur (5×5)', 'Máscara folha (Canny S)', 'Folha segmentada'],
    cols=4, figsize=(18, 5),
    nome='03_hsv_verde.png', results_dir=RD,
)

sep('Análise: Máscara da Folha (Canny canal S)')
total_px   = TAMANHO[0] * TAMANHO[1]
area_folha = int(mascara_folha.sum()) // 255
print('Pipeline: Blur 5×5 → HSV → Canny(S) → fillPoly → remove cinza → desconecta pecíolo → CLOSE')
print()
print(f'  Resolução       : {TAMANHO[0]}×{TAMANHO[1]} = {total_px:,} px')
print(f'  Área da folha   : {area_folha:,} px  ({area_folha/total_px*100:.1f}%)')
print(f'  Fundo           : {total_px-area_folha:,} px  ({(total_px-area_folha)/total_px*100:.1f}%)')
print()
print('Canal S distingue folha colorida (S alto) de fundo cinza/sombra (S ≈ 0).')
print('Erosão 25×25 desconecta o pecíolo (~20-30 px) mantendo só o corpo da folha.')


---
## 5 · Etapa 3b — HSV Marrom: Detecção das Lesões Black Rot

Entrada: `IMG_A_B` + `mascara_folha` (só busca dentro da folha).

Cor de referência medida nas manchas com conta-gotas: **R=35, G=16, B=7**
→ HSV OpenCV: H≈10, S≈200, V≈35 (marrom escuro, quase preto).

Faixas HSV combinadas (OR) — H ≤ 35 exclui amarelo-verde (H > 35 = 70°+ standard):

| Faixa | H | S | V | Alvo |
|---|---|---|---|---|
| Centro escuro/preto | 0..20 | 30..255 | 10..90 | Núcleo da lesão |
| Halo marrom escuro | 0..20 | 40..255 | 30..160 | Anel ao redor |
| Borda marrom-laranja | 20..35 | **80..255** | 30..110 | Halo externo — V<110 impede capturar tecido verde |

Pós-processamento: OPEN → CLOSE → connectedComponents (filtra áreas < 80 px, aspecto < 3).


In [ ]:
# HSV MARROM — detecta manchas Black Rot dentro da folha
mascara_lesao, img_lesao, n_lesoes, bboxes = detectar_lesoes_hsv(
    IMG_A_B,
    sensibilidade='grape_black_rot',
    mascara_folha=mascara_folha,
    area_minima=80,
    max_aspecto=3.0,
)

grade(
    [IMG_A_B,             mascara_lesao,                           img_lesao              ],
    ['Pré-Blur (entrada)', 'Máscara HSV marrom (lesões Black Rot)', 'Regiões doentes isoladas'],
    cols=3, figsize=(14, 5),
    nome='04_hsv_marrom.png', results_dir=RD,
)
print(f'Lesões detectadas pelo HSV: {n_lesoes}')

sep('Análise: HSV Marrom (lesões Black Rot)')
area_lesao = int(mascara_lesao.sum()) // 255
pct_lesao  = area_lesao / area_folha * 100 if area_folha > 0 else 0

hsv_xyxy = [(x, y, x+w, y+h) for (x, y, w, h) in bboxes]
gt_xyxy  = [(x1, y1, x2, y2) for (_, x1, y1, x2, y2) in GT_A]
m_hsv    = casar_caixas(gt_xyxy, hsv_xyxy, iou_min=0.2)

print(f'  Área lesionada      : {area_lesao:,} px  ({pct_lesao:.1f}% da folha)')
print(f'  Lesões detectadas   : {n_lesoes}')
print()
print(f'  GT boxes (real)     : {m_hsv["n_gt"]}')
print(f'  TP (IoU >= 0.2)     : {m_hsv["tp"]}')
print(f'  Recall              : {m_hsv["recall"]*100:.1f}%')
print(f'  Precision           : {m_hsv["precision"]*100:.1f}%')
print(f'  IoU médio           : {m_hsv["iou_medio"]:.2f}')
print()
print('Bounding boxes por lesão:')
for i, (x, y, w, h) in enumerate(bboxes):
    print(f'  Lesão {i+1:>2}: x={x:>3} y={y:>3} w={w:>3} h={h:>3}  área={w*h:>5} px')


---
## 6 · Etapa 4 — Canny por ROI: Refinamento de Bordas

Canny aplicado **somente dentro das ROIs** já localizadas pelo HSV.

**Por que isso melhora:**
- Canny na imagem inteira detecta nervuras (falso positivo de estrutura)
- Canny na ROI da lesão detecta apenas a borda da mancha
- Thresholds automáticos pela mediana da ROI (mais precisos por região)
- Permite medir circularidade e irregularidade de cada lesão


In [ ]:
# CANNY POR ROI — refinamento de borda em cada lesão detectada pelo HSV
img_canny_lesoes = IMG_A.copy()
_h_img, _w_img = IMG_A.shape[:2]
_n_borda_total  = 0

for (x, y, w, h) in bboxes:
    _x0, _y0 = max(0, x - 4), max(0, y - 4)
    _x1, _y1 = min(_w_img, x + w + 4), min(_h_img, y + h + 4)
    _roi = converter_grayscale(IMG_A_B[_y0:_y1, _x0:_x1])  # já blurrada
    _med = float(np.median(_roi))
    _b   = cv2.Canny(_roi, int(max(0, 0.66 * _med)), int(min(255, 1.33 * _med)))
    img_canny_lesoes[_y0:_y1, _x0:_x1][_b > 0] = [0, 220, 220]  # ciano
    _n_borda_total += int((_b > 0).sum())

grade(
    [IMG_A,      mascara_lesao,            img_canny_lesoes            ],
    ['Original', 'Máscara HSV marrom',     'Canny por ROI (ciano)'],
    cols=3, figsize=(14, 5),
    nome='05_canny_roi.png', results_dir=RD,
)

sep('Análise: Canny por ROI')
print(f'  Lesões HSV             : {n_lesoes}')
print(f'  Pixels borda (total)   : {_n_borda_total:,}')
if bboxes:
    _areas = [w*h for (x,y,w,h) in bboxes]
    print(f'  Área média por ROI     : {int(np.mean(_areas))} px')
    print(f'  Área mín / máx         : {min(_areas)} / {max(_areas)} px')
print()
print('Bordas detectadas pertencem APENAS às lesões (nervuras excluídas).')
print('Aplicação: medir circularidade → lesões circulares = estágio inicial.')
print('            borda irregular → estágio avançado / coalescing.')


---
## 7 · Diagnóstico Visual Final

In [ ]:
# Blend vermelho + bboxes + rodapé informativo
diagnostico  = compor_diagnostico_visual(IMG_A, mascara_lesao, bboxes, NOME_A)
img_gt       = desenhar_boxes(IMG_A, GT_A, CLASSES)

blend_amarelo = destacar_regiao_doenca(
    IMG_A, mascara_lesao, cor_destaque=(255, 220, 0), alpha=0.55)

grade(
    [img_gt,                            diagnostico,                   blend_amarelo        ],
    [f'Ground-Truth ({len(GT_A)} GT)',  f'Diagnóstico ({n_lesoes} lesões)', 'Blend amarelo'],
    cols=3, figsize=(15, 5),
    nome='06_diagnostico_final.png', results_dir=RD,
)

sep('Resumo do Pipeline — Grape Black Rot')
print('  Blur 5×5 → HSV verde (folha) → HSV marrom (lesão) → Canny por ROI')
print()
print(f'  {"Amostra":<22}: {os.path.basename(PATH_A)}')
print(f'  {"Classe":<22}: {NOME_A}')
print(f'  {"GT boxes":<22}: {len(GT_A)}')
print(f'  {"Lesões HSV":<22}: {n_lesoes}')
print(f'  {"Recall":<22}: {m_hsv["recall"]*100:.1f}%')
print(f'  {"Precision":<22}: {m_hsv["precision"]*100:.1f}%')
print(f'  {"IoU médio":<22}: {m_hsv["iou_medio"]:.2f}')
print(f'  {"Pixels borda (Canny)":<22}: {_n_borda_total:,}')


---
## 8 · Grid — Resultados do Pipeline em Múltiplas Imagens

Pipeline completo aplicado a 6 amostras de **Grape Black Rot** do conjunto de treino.
Cada painel mostra o diagnóstico final: blend vermelho + bounding boxes detectados.


In [ ]:
# Filtrar imagens de Grape Black Rot pelo índice de classe
GRAPE_CLASS_IDX = next(i for i, c in enumerate(CLASSES) if 'grape' in c.lower() and 'rot' in c.lower())
_pool_grid = amostrar_imagens(DATA_DIR, n=300, split='train', com_label=True, semente=7)

amostras_gbr = []
for _p in _pool_grid:
    _img_tmp = cv2.resize(carregar_imagem_rgb(_p), TAMANHO)
    _gt_tmp  = ler_labels_yolo(_p, _img_tmp.shape)
    if any(c[0] == GRAPE_CLASS_IDX for c in _gt_tmp):
        amostras_gbr.append((_p, _img_tmp, _gt_tmp))
    if len(amostras_gbr) >= 6:
        break

print(f'Grape Black Rot encontradas: {len(amostras_gbr)}')

imgs_grid, titulos_grid = [], []
for _caminho, _img_raw, _gt in amostras_gbr:
    _img_b  = aplicar_blur_gaussiano(_img_raw, kernel=(5, 5))
    _, _mf  = segmentar_folha_verde(_img_b)
    _ml, _, _nles, _bbs = detectar_lesoes_hsv(
        _img_b, sensibilidade='grape_black_rot',
        mascara_folha=_mf, area_minima=80, max_aspecto=3.0,
    )
    _diag = compor_diagnostico_visual(_img_raw, _ml, _bbs, 'Grape Black Rot')
    imgs_grid.append(_diag)
    titulos_grid.append(f'GT={len(_gt)}  Det={_nles}')

grade(
    imgs_grid, titulos_grid,
    cols=3, figsize=(18, 12),
    nome='07_grid_multiplas.png', results_dir=RD,
)
print('Grid salvo em 07_grid_multiplas.png')


---
## 9 · Comparação: HSV Clássico vs YOLOv8 (fine-tuned)

| Método | Abordagem | Treinamento |
|---|---|---|
| **HSV Pipeline** | Regras manuais de cor + morfologia | Nenhum — baseado em domínio |
| **YOLOv8** | Rede neural convolucional | Fine-tune no dataset PlantVillage |

Comparação visual (Original | HSV | YOLO) e métricas (Recall, Precision, IoU) para 3 imagens.


In [ ]:
from yolo_detection import carregar_modelo_deteccao, detectar_objetos
from yolo_detection import comparar_metodos as _comparar_metodos

modelo_yolo = carregar_modelo_deteccao('plant_disease_best.pt')
print('Modelo carregado:', type(modelo_yolo).__name__, '| classes:', len(modelo_yolo.names))

_imgs_comp = amostras_gbr[:3] if len(amostras_gbr) >= 3 else amostras_gbr
_comparacoes, _resultados = [], []

for _caminho, _img_raw, _gt in _imgs_comp:
    _img_b = aplicar_blur_gaussiano(_img_raw, kernel=(5, 5))
    _, _mf = segmentar_folha_verde(_img_b)
    _, _, _nles, _bbs_hsv = detectar_lesoes_hsv(
        _img_b, sensibilidade='grape_black_rot',
        mascara_folha=_mf, area_minima=80, max_aspecto=3.0,
    )
    # Imagem HSV com bboxes em laranja
    _hsv_img = _img_raw.copy()
    for (x, y, w, h) in _bbs_hsv:
        cv2.rectangle(_hsv_img, (x, y), (x + w, y + h), (220, 80, 0), 2)

    # YOLO
    _yolo_img, _dets_yolo = detectar_objetos(_img_raw, modelo=modelo_yolo)

    # Painel comparativo
    _comp = _comparar_metodos(
        _img_raw, _hsv_img, _yolo_img,
        label_classico='HSV Pipeline', label_yolo='YOLOv8 (fine-tuned)',
    )
    _comparacoes.append(_comp)

    # Métricas
    _gt_xyxy   = [(x1, y1, x2, y2) for (_, x1, y1, x2, y2) in _gt]
    _hsv_xyxy  = [(x, y, x + w, y + h) for (x, y, w, h) in _bbs_hsv]
    _yolo_xyxy = [d['bbox'] for d in _dets_yolo]
    _mh = casar_caixas(_gt_xyxy, _hsv_xyxy,  iou_min=0.2)
    _my = casar_caixas(_gt_xyxy, _yolo_xyxy, iou_min=0.2)
    _resultados.append({
        'nome':  os.path.basename(_caminho)[:22],
        'gt':    len(_gt),
        'hsv':   _mh,
        'yolo':  _my,
    })

# Exibe e salva painel
_painel = np.vstack(_comparacoes)
plt.figure(figsize=(16, 5 * len(_comparacoes)))
plt.imshow(_painel)
plt.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(RD, '08_comparacao_hsv_yolo.png'), dpi=80, bbox_inches='tight')
plt.show()
plt.close()

sep('Métricas: HSV Clássico vs YOLOv8 fine-tuned')
_hdr = f'  {"Imagem":<24}  {"GT":>4}  {"Método":<18}  {"Det":>5}  {"TP":>4}  {"Recall":>8}  {"Prec":>8}  {"IoU":>6}'
print(_hdr)
print('-' * len(_hdr))
for _r in _resultados:
    for _label, _m in [('HSV Pipeline', _r['hsv']), ('YOLOv8 fine-tuned', _r['yolo'])]:
        print(
            f'  {_r["nome"]:<24}  {_r["gt"]:>4}  {_label:<18}  '
            f'{_m["n_pred"]:>5}  {_m["tp"]:>4}  '
            f'{_m["recall"]*100:>7.1f}%  {_m["precision"]*100:>7.1f}%  '
            f'{_m["iou_medio"]:>6.2f}'
        )
    print()


---
## 10 · Arquivos gerados

In [ ]:
print(f'  {"Arquivo":<44} {"Tamanho":>10}')
print('-' * 56)
for f in sorted(f for f in os.listdir(RD) if f.endswith('.png')):
    tam = os.path.getsize(os.path.join(RD, f)) / 1024
    print(f'  {f:<44} {tam:>6.1f} KB')
